# Automatic False-Positive Cleanup — Local Run

Run this **after** Step 4 of `main_local.ipynb` has produced detection CSVs in
`data/outputs/detections/`. It automatically sorts those detections into
**clean** (trust them) and **suspicious** (likely false positives) — **no manual
listening required**.

## What it does

Three independent filters score every detection:

1. **Mahalanobis OOD** — how far the detection's sound “looks” from real
   training examples of its predicted species.
2. **YAMNet cross-check** — Google's audio tagger flags windows it thinks are
   Bird / Insect / Wind / Rain / Speech rather than a primate call.
3. **Temporal isolation** — a call with no same-species neighbour within ±30 s
   is suspicious (primates call in bouts, not once).

A detection is **clean** only if all three filters agree it's real. Anything
flagged by **≥2** filters is saved as a hard negative for the next retraining
round. The logic lives in `src/auto_cleanup.py`; this notebook just drives it.

## Setup (one time)

Same environment as `main_local.ipynb` — if you already ran that notebook you're
ready. Otherwise follow [`SETUP.md`](../SETUP.md). Then just run the cells below
**in order**.

## Step 1 — Set up paths and load config

Locates the repo root automatically and points the pipeline at this repo's
`data/` folder, exactly like `main_local.ipynb`. Selects the production V12 head.

In [ ]:
import os, sys, importlib
from pathlib import Path

def find_repo_root(start=None):
    """Walk up from the working directory until we find src/config.py + data/."""
    p = Path(start or os.getcwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / 'src' / 'config.py').is_file() and (cand / 'data').is_dir():
            return cand
    raise RuntimeError('Run this notebook from inside the cloned repo '
                       '(it needs src/config.py and the data/ folder).')

REPO = find_repo_root()
os.environ['PRIMATE_DATA_ROOT']     = str(REPO / 'data')
os.environ['PRIMATE_MODEL_POOLING'] = 'temporal_freqpos'   # production V12 head

SRC = str(REPO / 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import config
importlib.reload(config)
print('Repo root      :', REPO)
print('Data root      :', config.DRIVE_ROOT)
print('Detections dir :', config.DETECTION_OUTPUT_DIR)

## Step 1.5 — GPU check (automatic)

TensorFlow uses a GPU if one is available; otherwise it runs on CPU (fine for
this — cleanup is mostly feature extraction). This cell just reports what you have.

In [ ]:
import platform, tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'GPU(s) found: {len(gpus)} — TensorFlow will use GPU automatically.')
else:
    print(f'No GPU detected (running on CPU — {platform.system()} {platform.machine()}).')
    print('That is fine for cleanup; it is just a little slower.')

print(f'\nTensorFlow {tf.__version__} | Python {platform.python_version()}')

## Step 2 — Load the trained model

Loads `data/outputs/models/best_model_v12.h5` (the same model `main_local.ipynb`
uses). The V12 head has a custom `FrequencyCoord` layer, so it is loaded with
`model.load_trained_model(...)`, not raw `tf.keras.models.load_model(...)`.

In [ ]:
import model as model_lib
importlib.reload(model_lib)

candidates = ['best_model_v12.h5', 'best_model.h5']
model_path = next(
    (os.path.join(config.MODEL_SAVE_DIR, n)
     for n in candidates if os.path.exists(os.path.join(config.MODEL_SAVE_DIR, n))),
    None)
assert model_path, ('No model found. Put best_model_v12.h5 in '
                    'data/outputs/models/  (see main_local.ipynb Step 3).')

print('Loading model:', model_path)
model = model_lib.load_trained_model(model_path)
print('Model loaded.')

## Step 3 — Run the auto-cleanup

By default this reads every `*_detections.csv` under
`data/outputs/detections/` (searched recursively, so per-station subfolders
work). To clean a specific folder only, set `DETECTION_DIR` to that path;
leave it as `None` to use the default.

Results (clean / suspicious CSVs + hard-negative clips) are written under
`data/outputs/auto_cleanup/`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import auto_cleanup
importlib.reload(auto_cleanup)

# None = clean everything in data/outputs/detections/. Or point at one folder:
# DETECTION_DIR = os.path.join(config.DETECTION_OUTPUT_DIR, 'IPA1ST')
DETECTION_DIR = None

result = auto_cleanup.run_auto_cleanup(model=model, detection_dir=DETECTION_DIR)
det_df       = result['det_df']
CLEANUP_DIR  = Path(result['output_dir'])

print('\nVerdict per species:')
result['summary']

## Step 4 — Visualise the verdict

Left: how many detections per species were clean vs flagged. Right: which
combination of filters did the flagging. Saved to
`data/outputs/auto_cleanup/auto_cleanup_summary.png`.

In [ ]:
if len(det_df) == 0:
    print('No detections found. Run Step 4 of main_local.ipynb first, '
          'or check DETECTION_DIR.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    # (a) stacked bar: clean / 1-flag / 2-flags / 3-flags per species
    order = [s for s in config.CLASS_NAMES if s in det_df['species'].unique()]
    counts = pd.DataFrame({
        'clean':   [int(((det_df.species == s) & (det_df.n_flags == 0)).sum()) for s in order],
        '1 flag':  [int(((det_df.species == s) & (det_df.n_flags == 1)).sum()) for s in order],
        '2 flags': [int(((det_df.species == s) & (det_df.n_flags == 2)).sum()) for s in order],
        '3 flags': [int(((det_df.species == s) & (det_df.n_flags == 3)).sum()) for s in order],
    }, index=order)
    colors = ['#27AE60', '#F1C40F', '#E67E22', '#C0392B']
    counts.plot(kind='bar', stacked=True, color=colors, ax=axes[0],
                edgecolor='black', linewidth=0.5)
    axes[0].set_title('Auto-cleanup verdict per species')
    axes[0].set_ylabel('Detections')
    axes[0].tick_params(axis='x', rotation=15)
    axes[0].legend(fontsize=9)

    # (b) which filters agreed
    venn_rows = {
        'Mahal only':    int(((det_df.flag_mahal) & (~det_df.flag_yamnet) & (~det_df.flag_isolated)).sum()),
        'YAMNet only':   int(((~det_df.flag_mahal) & (det_df.flag_yamnet) & (~det_df.flag_isolated)).sum()),
        'Isolated only': int(((~det_df.flag_mahal) & (~det_df.flag_yamnet) & (det_df.flag_isolated)).sum()),
        'M+Y':           int(((det_df.flag_mahal) & (det_df.flag_yamnet) & (~det_df.flag_isolated)).sum()),
        'M+I':           int(((det_df.flag_mahal) & (~det_df.flag_yamnet) & (det_df.flag_isolated)).sum()),
        'Y+I':           int(((~det_df.flag_mahal) & (det_df.flag_yamnet) & (det_df.flag_isolated)).sum()),
        'All 3':         int(((det_df.flag_mahal) & (det_df.flag_yamnet) & (det_df.flag_isolated)).sum()),
    }
    axes[1].bar(list(venn_rows.keys()), list(venn_rows.values()),
                color=['#F1C40F', '#F1C40F', '#F1C40F', '#E67E22', '#E67E22', '#E67E22', '#C0392B'],
                edgecolor='black', linewidth=0.6)
    axes[1].set_title('Filter agreement')
    axes[1].set_ylabel('Detections')
    axes[1].tick_params(axis='x', rotation=25)

    plt.tight_layout()
    out = CLEANUP_DIR / 'auto_cleanup_summary.png'
    plt.savefig(out, dpi=180, bbox_inches='tight')
    plt.show()
    print(f'Summary chart saved to {out}')

## Done

Outputs are under `data/outputs/auto_cleanup/`:

- `clean_detections.csv` — passed all three filters; trust without listening
- `suspicious_detections.csv` — flagged, with a `flag_reason` column
- `auto_flagged_fp/<reason>/*.wav` — ≥2-flag clips to fold into Background
- `auto_cleanup_summary.png` — the chart above

**To improve the model next round:** copy everything under `auto_flagged_fp/`
into a new background folder (e.g. `data/background/auto_hard_negatives/`), add
that folder to `config.BACKGROUND_FOLDERS`, then re-run `main_local.ipynb`
Step 3 with `FORCE_RETRAIN = True`. Each iteration should shrink the
false-positive set.

**Command-line equivalent:**

```
python scripts/run_auto_cleanup.py --detection-dir data/outputs/detections
```